In [44]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('multi_stock_dataset.csv')
df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 17094 entries, 0 to 17093
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    17094 non-null  str    
 1   Stock   17094 non-null  str    
 2   Open    16443 non-null  float64
 3   High    16443 non-null  float64
 4   Low     16443 non-null  float64
 5   Close   16443 non-null  float64
 6   Volume  16443 non-null  float64
dtypes: float64(5), str(2)
memory usage: 935.0 KB


,Open,High,Low,Close,Volume
count,16443.000000,16443.000000,16443.000000,16443.000000,1.644300e+04
mean,539.358234,544.732998,533.702447,539.184766,4.438367e+07
std,729.130566,735.771052,721.936211,728.839995,7.558099e+07
min,11.796331,12.149944,11.709067,11.874168,0.000000e+00
25%,54.740819,55.289033,54.217394,54.727184,6.889784e+06
50%,211.944270,214.002034,209.708079,211.977692,2.122080e+07
75%,824.701042,832.270741,812.892446,825.592194,4.370519e+07
max,3593.135350,3659.166023,3567.104704,3595.836426,1.460852e+09


In [45]:
df.head()

,Date,Stock,Open,High,Low,Close,Volume
0,2013-01-01,AAPL,NaN,NaN,NaN,NaN,NaN
1,2013-01-02,AAPL,16.741478,16.777148,16.372986,16.596680,560518000.0
2,2013-01-03,AAPL,16.561914,16.616024,16.353938,16.387190,352965200.0
3,2013-01-04,AAPL,16.232120,16.282300,15.895367,15.930736,594333600.0
4,2013-01-07,AAPL,15.779593,16.000265,15.574035,15.837029,484156400.0


In [46]:
df.info()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 17094 entries, 0 to 17093
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    17094 non-null  str    
 1   Stock   17094 non-null  str    
 2   Open    16443 non-null  float64
 3   High    16443 non-null  float64
 4   Low     16443 non-null  float64
 5   Close   16443 non-null  float64
 6   Volume  16443 non-null  float64
dtypes: float64(5), str(2)
memory usage: 935.0 KB


Date        0
Stock       0
Open      651
High      651
Low       651
Close     651
Volume    651
dtype: int64

In [47]:
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Stock', 'Date'])
df.info()
df.head()

<class 'pandas.DataFrame'>
Index: 17094 entries, 0 to 14244
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    17094 non-null  datetime64[us]
 1   Stock   17094 non-null  str           
 2   Open    16443 non-null  float64       
 3   High    16443 non-null  float64       
 4   Low     16443 non-null  float64       
 5   Close   16443 non-null  float64       
 6   Volume  16443 non-null  float64       
dtypes: datetime64[us](1), float64(5), str(1)
memory usage: 1.0 MB


,Date,Stock,Open,High,Low,Close,Volume
0,2013-01-01,AAPL,NaN,NaN,NaN,NaN,NaN
1,2013-01-02,AAPL,16.741478,16.777148,16.372986,16.596680,560518000.0
2,2013-01-03,AAPL,16.561914,16.616024,16.353938,16.387190,352965200.0
3,2013-01-04,AAPL,16.232120,16.282300,15.895367,15.930736,594333600.0
4,2013-01-07,AAPL,15.779593,16.000265,15.574035,15.837029,484156400.0


In [48]:
# 1. Handle missing values
# Forward fill then backward fill within each stock to prevent data leakage across stocks
for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
    df[col] = df.groupby('Stock')[col].ffill()
    df[col] = df.groupby('Stock')[col].bfill()

df.isnull().sum()

Date      0
Stock     0
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64

In [49]:
# 2. Feature Engineering
# Calculate daily returns
df['Daily_Return'] = df.groupby('Stock')['Close'].pct_change()

# Calculate Moving Averages (MA)
df['MA_10'] = df.groupby('Stock')['Close'].transform(lambda x: x.rolling(window=10).mean())
df['MA_50'] = df.groupby('Stock')['Close'].transform(lambda x: x.rolling(window=50).mean())

# Calculate Volatility (rolling standard deviation of returns)
df['Volatility_10'] = df.groupby('Stock')['Daily_Return'].transform(lambda x: x.rolling(window=10).std())

df.head(15)

,Date,Stock,Open,High,Low,Close,Volume,Daily_Return,MA_10,MA_50,Volatility_10
0,2013-01-01,AAPL,16.741478,16.777148,16.372986,16.596680,560518000.0,NaN,NaN,NaN,NaN
1,2013-01-02,AAPL,16.741478,16.777148,16.372986,16.596680,560518000.0,0.000000,NaN,NaN,NaN
2,2013-01-03,AAPL,16.561914,16.616024,16.353938,16.387190,352965200.0,-0.012622,NaN,NaN,NaN
3,2013-01-04,AAPL,16.232120,16.282300,15.895367,15.930736,594333600.0,-0.027854,NaN,NaN,NaN
4,2013-01-07,AAPL,15.779593,16.000265,15.574035,15.837029,484156400.0,-0.005882,NaN,NaN,NaN
5,2013-01-08,AAPL,15.997541,16.078555,15.756918,15.879647,458707200.0,0.002691,NaN,NaN,NaN
6,2013-01-09,AAPL,15.794698,15.870574,15.597907,15.631462,407604400.0,-0.015629,NaN,NaN,NaN
7,2013-01-10,AAPL,15.977588,15.982726,15.583703,15.825234,601146000.0,0.012396,NaN,NaN,NaN
8,2013-01-11,AAPL,15.749359,15.879949,15.689505,15.728199,350506800.0,-0.006132,NaN,NaN,NaN
9,2013-01-14,AAPL,15.195558,15.341263,15.069503,15.167446,734207600.0,-0.035653,15.958030,NaN,NaN


In [50]:
# 3. Drop rows with NaNs introduced by rolling windows
df.dropna(inplace=True)
df.isnull().sum()

Date             0
Stock            0
Open             0
High             0
Low              0
Close            0
Volume           0
Daily_Return     0
MA_10            0
MA_50            0
Volatility_10    0
dtype: int64

In [52]:
# 4. Encoding categorical features and scaling
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

# Encode Stock ticker
le = LabelEncoder()
df['Stock_Encoded'] = le.fit_transform(df['Stock'])

# Scale numeric features
scaler = MinMaxScaler()
cols_to_scale = ['Open', 'High', 'Low', 'Close', 'Volume', 'Daily_Return', 'MA_10', 'MA_50', 'Volatility_10']

df_scaled = df.copy()
df_scaled[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

df_scaled.head()

,Date,Stock,Open,High,Low,Close,Volume,Daily_Return,MA_10,MA_50,Volatility_10,Stock_Encoded
49,2013-03-11,AAPL,0.000355,0.000329,0.000342,0.000402,0.445073,0.604609,0.000236,0.000363,0.173688,0
50,2013-03-12,AAPL,0.000404,0.000327,0.000363,0.000322,0.437261,0.509174,0.000219,0.000342,0.176364,0
51,2013-03-13,AAPL,0.000344,0.000291,0.000344,0.000321,0.380610,0.566141,0.000205,0.000320,0.175720,0
52,2013-03-14,AAPL,0.000381,0.000292,0.000388,0.000356,0.285189,0.592462,0.000197,0.000301,0.180463,0
53,2013-03-15,AAPL,0.000424,0.000372,0.000446,0.000451,0.604361,0.635416,0.000208,0.000286,0.180043,0


In [53]:
print(df)

            Date   Stock         Open         High          Low        Close  \
49    2013-03-11    AAPL    13.066666    13.348219    12.926497    13.313557   
50    2013-03-12    AAPL    13.244536    13.344265    13.000381    13.026529   
51    2013-03-13    AAPL    13.027141    13.211092    12.933188    13.024099   
52    2013-03-14    AAPL    13.160312    13.215345    13.087947    13.150279   
53    2013-03-15    AAPL    13.315380    13.506933    13.294704    13.489602   
...          ...     ...          ...          ...          ...          ...   
14240 2023-12-22  TCS.NS  3539.025215  3581.819435  3503.634963  3561.376953   
14241 2023-12-26  TCS.NS  3557.512040  3570.690161  3529.851582  3534.880859   
14242 2023-12-27  TCS.NS  3538.093966  3555.975312  3509.222971  3549.456055   
14243 2023-12-28  TCS.NS  3561.377091  3574.415606  3531.667996  3538.932129   
14244 2023-12-29  TCS.NS  3531.574902  3560.073461  3506.801632  3532.878662   

            Volume  Daily_Return       